<a href="https://colab.research.google.com/github/debolton/MusicAnnotationProject/blob/main/MusicAnnotationProject_V3_02_GPU_Local_Staging_Train_Evaluate_GST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music Annotation Project V3.2: GPU Local-Staging Training, Validation and GST Comparison

**Runtime:** T4 GPU. Select **Runtime > Run all**.

This notebook:

1. validates the V3 dataset before spending GPU time;
2. rejects a run if manifest page names do not exactly match page prefixes present in the train and validation image folders;
3. stages required inputs on the Colab local server;
4. trains the configured YOLO segmentation model from local storage;
4. evaluates Page022 and Page023 quantitatively;
5. saves tiled and full-page validation predictions;
6. runs original, background-normalised and CLAHE GST tiled inference;
7. displays three-way GST comparisons;
8. saves page-level and instance-level GST diagnostics plus one combined run summary.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 11.6 MB/s eta 0:00:00


## Load configuration and perform the pre-training safety gate

The safety gate checks:

- the manifest and YAML exist;
- validation pages match the configuration;
- train and validation pages do not overlap;
- image and label filenames pair exactly;
- page prefixes in dataset images exactly match the manifest page lists;
- no validation page appears in the training image folder;
- a CUDA GPU is available.

If any check fails, training does not start.

In [ ]:
from os import truncate
import cv2, json, hashlib, csv, re, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

CONFIG_PATH=Path('/content/drive/MyDrive/MusicAnnotationProject/project_config.json')
if not CONFIG_PATH.exists():
    raise RuntimeError(f'Missing configuration file: {CONFIG_PATH}')

CONFIG=json.loads(CONFIG_PATH.read_text())
ROOT=Path(CONFIG['project_root'])
DATASET=ROOT/'dataset'
YAML=ROOT/'dataset_v3.yaml'
MANIFEST=ROOT/'dataset_manifest_v3.json'
GOLD=ROOT/'gold_standard_testing'
RUNS=ROOT/'training_runs'
PREDS=ROOT/'predictions'
ALIGNED=ROOT/'aligned'
T=CONFIG['training']
FIXED_VALIDATION=set(CONFIG['validation_pages'])
EXTENSIONS={'.jpg','.jpeg','.png','.tif','.tiff'}


# Local-server staging controls
STAGE_DATASET_LOCALLY=True
REBUILD_DRIVE_ARCHIVE=True
DRIVE_DATA_ARCHIVE=ROOT/'staging'/'yolo_training_inputs.zip'
LOCAL_ROOT=Path('/content/MusicAnnotationProject_local')
LOCAL_DATASET=LOCAL_ROOT/'dataset'
LOCAL_YAML=LOCAL_ROOT/'dataset_v3_local.yaml'
LOCAL_RUNS=LOCAL_ROOT/'training_runs'
LOCAL_PREDS=LOCAL_ROOT/'predictions'
LOCAL_ALIGNED=LOCAL_ROOT/'aligned'
LOCAL_GOLD=LOCAL_ROOT/'gold_standard_testing'

# Preserve durable Google Drive destinations before switching compute paths.
DRIVE_DATASET=DATASET
DRIVE_MANIFEST=MANIFEST
DRIVE_GOLD=GOLD
DRIVE_RUNS=RUNS
DRIVE_PREDS=PREDS
DRIVE_ALIGNED=ALIGNED

# Standard comparison is included in this notebook.
RUN_GST_NORMALISATION_COMPARISON=True
BACKGROUND_BLUR_SIGMA=75
CLAHE_CLIP_LIMIT=2.0
CLAHE_GRID_SIZE=(8,8)
GST_TILE_SIZE=int(CONFIG['gold_standard']['tile_size'])
GST_TILE_STRIDE=int(CONFIG['gold_standard']['tile_stride'])
GST_CONFIDENCE=float(T['prediction_confidence'])
GST_IMAGE_SIZE=int(T['image_size'])


def images(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS) if folder.exists() else []


def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()


def page_prefix_from_tile(stem):
    match=re.match(r'^(.*)_y\d+_x\d+$',stem)
    return match.group(1) if match else None


if not YAML.exists() or not MANIFEST.exists():
    raise RuntimeError('Run the V3.1 dataset notebook first.')

manifest=json.loads(MANIFEST.read_text())
train_pages=set(manifest['training_pages'])
val_pages=set(manifest['validation_pages'])

# The manifest is the complete source of truth.
# project_config.json now identifies only the
# fixed validation pages that must always remain
# in validation.

missing_fixed_validation = (
    FIXED_VALIDATION
    - val_pages
)

if missing_fixed_validation:
    raise RuntimeError(f'Fixed validation pages missing from manifest: {missing_fixed_validation}')

if train_pages & val_pages:
    raise RuntimeError(f'Train/validation page overlap: {sorted(train_pages & val_pages)}')

print(
    "Fixed validation pages:",
    sorted(FIXED_VALIDATION),
)

print(
    "Complete manifest validation pages:",
    sorted(val_pages),
)

folder_page_prefixes={}
for split,manifest_pages in [('train',train_pages),('val',val_pages)]:
    image_dir=DATASET/'images'/split
    label_dir=DATASET/'labels'/split
    image_map={p.stem:p for p in images(image_dir)}
    label_map={p.stem:p for p in label_dir.glob('*.txt')} if label_dir.exists() else {}
    if set(image_map)!=set(label_map):
        raise RuntimeError(
            f'{split}: image/label mismatch. '
            f'Missing labels={sorted(set(image_map)-set(label_map))[:20]}; '
            f'missing images={sorted(set(label_map)-set(image_map))[:20]}'
        )
    bad_names=[stem for stem in image_map if page_prefix_from_tile(stem) is None]
    if bad_names:
        raise RuntimeError(f'{split}: tile names do not match PageName_y####_x####: {bad_names[:20]}')
    prefixes={page_prefix_from_tile(stem) for stem in image_map}
    folder_page_prefixes[split]=prefixes
    missing_from_folder=manifest_pages-prefixes
    unlisted_in_manifest=prefixes-manifest_pages
    if missing_from_folder or unlisted_in_manifest:
        raise RuntimeError(
            f'{split}: manifest/page-prefix mismatch. '
            f'Manifest pages absent from image folder={sorted(missing_from_folder)}; '
            f'image-folder pages absent from manifest={sorted(unlisted_in_manifest)}'
        )
    print(f'{split}: {len(image_map)} image/label pairs across {len(prefixes)} page prefixes.')

if folder_page_prefixes['train'] & folder_page_prefixes['val']:
    raise RuntimeError('Page prefixes occur in both training and validation image folders.')
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then Run all again.')

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+Path(T['model']).stem
RUN_DIR=RUNS/RUN
RUN_PRED=PREDS/RUN

print('GPU:',torch.cuda.get_device_name(0))
print('Run:',RUN)
print('Manifest training page count:',len(train_pages))
print('Dataset training prefix count:',len(folder_page_prefixes['train']))
print('Training pages:',sorted(train_pages))
print('Validation pages:',sorted(val_pages))
print('Gold files:',[p.name for p in images(GOLD)])
print('PRE-TRAINING SAFETY GATE PASSED')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Fixed validation pages: ['Page009', 'Page022', 'Page023', 'Page025', 'Page033', 'Page061', 'Page068', 'Page074', 'Page080', 'Page085']
Complete manifest validation pages: ['Page009', 'Page022', 'Page023', 'Page025', 'Page033', 'Page061', 'Page068', 'Page074', 'Page080', 'Page085', 'Page091', 'Page096', 'Page101', 'Page106', 'Page111', 'Page116', 'Page121', 'Page126', 'Page131', 'Page136', 'Page141', 'Page146', 'Page151', 'Page156', 'Page161', 'Page166', 'Page171', 'Page176', 'Page181', 'Page186', 'Page191', 'Page196']
train: 4645 image/label pairs across 163 page prefixes.
val: 923 image/label pairs across 32 page prefixes.
GPU: NVIDIA L4
Run: 20261007_171755_yolo11s-seg
Manifest training page coun

## Stage the required inputs on the Colab local server

The safety gate runs against Google Drive first. This step then creates an uncompressed ZIP containing only the prepared dataset, manifest, configuration, aligned validation pages and GST images. The ZIP is copied to `/content` and extracted.

Training, validation, GST inference and run-output writing use local storage. Completed training and prediction folders are copied back to Google Drive at the end.

`REBUILD_DRIVE_ARCHIVE=True` guarantees that each run uses the current dataset. Change it to `False` only when deliberately reusing an archive created from an unchanged dataset.

In [ ]:
if STAGE_DATASET_LOCALLY:
    DRIVE_DATA_ARCHIVE.parent.mkdir(parents=True,exist_ok=True)
    required_sources={
        'dataset':DRIVE_DATASET,
        'dataset_manifest_v3.json':DRIVE_MANIFEST,
        'project_config.json':CONFIG_PATH,
        'gold_standard_testing':DRIVE_GOLD,
        'aligned':DRIVE_ALIGNED,
    }

    if REBUILD_DRIVE_ARCHIVE or not DRIVE_DATA_ARCHIVE.exists():
        temporary_archive=Path('/content/yolo_training_inputs_build.zip')
        if temporary_archive.exists(): temporary_archive.unlink()
        print('Creating a fresh staging ZIP from Google Drive...')
        with zipfile.ZipFile(temporary_archive,'w',compression=zipfile.ZIP_STORED) as archive:
            for archive_name,source in required_sources.items():
                source=Path(source)
                if not source.exists(): raise RuntimeError(f'Required staging source missing: {source}')
                if source.is_file():
                    archive.write(source,arcname=archive_name)
                else:
                    for path in source.rglob('*'):
                        if path.is_file() and not path.name.endswith('.cache'):
                            archive.write(path,arcname=str(Path(archive_name)/path.relative_to(source)))
        shutil.copy2(temporary_archive,DRIVE_DATA_ARCHIVE)
        print('Drive staging archive refreshed:',DRIVE_DATA_ARCHIVE)
    else:
        print('Reusing Drive staging archive:',DRIVE_DATA_ARCHIVE)

    local_archive=Path('/content/yolo_training_inputs.zip')
    if local_archive.exists(): local_archive.unlink()
    shutil.copy2(DRIVE_DATA_ARCHIVE,local_archive)
    print('Archive copied locally:',round(local_archive.stat().st_size/1024/1024,1),'MB')

    if LOCAL_ROOT.exists(): shutil.rmtree(LOCAL_ROOT)
    LOCAL_ROOT.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(local_archive,'r') as archive:
        archive.extractall(LOCAL_ROOT)

    LOCAL_RUNS.mkdir(parents=True,exist_ok=True)
    LOCAL_PREDS.mkdir(parents=True,exist_ok=True)
    LOCAL_YAML.write_text(
        f'path: {LOCAL_DATASET}\n'
        'train: images/train\n'
        'val: images/val\n'
        'names:\n'
        '  0: annotation\n'
    )

    DATASET=LOCAL_DATASET
    YAML=LOCAL_YAML
    GOLD=LOCAL_GOLD
    ALIGNED=LOCAL_ALIGNED
    RUNS=LOCAL_RUNS
    PREDS=LOCAL_PREDS
    RUN_DIR=RUNS/RUN
    RUN_PRED=PREDS/RUN

    local_train_images=images(DATASET/'images'/'train')
    local_val_images=images(DATASET/'images'/'val')
    local_train_labels=list((DATASET/'labels'/'train').glob('*.txt'))
    local_val_labels=list((DATASET/'labels'/'val').glob('*.txt'))
    if len(local_train_images)!=len(local_train_labels): raise RuntimeError('Local train image/label count mismatch.')
    if len(local_val_images)!=len(local_val_labels): raise RuntimeError('Local validation image/label count mismatch.')
    if not local_train_images or not local_val_images: raise RuntimeError('Local staging produced an empty split.')

    print('Local training tiles:',len(local_train_images))
    print('Local validation tiles:',len(local_val_images))
    print('Local YAML:',YAML)
    print('LOCAL STAGING PASSED')

Creating a fresh staging ZIP from Google Drive...
Drive staging archive refreshed: /content/drive/MyDrive/MusicAnnotationProject/staging/yolo_training_inputs.zip
Archive copied locally: 7886.3 MB
Local training tiles: 4645
Local validation tiles: 923
Local YAML: /content/MusicAnnotationProject_local/dataset_v3_local.yaml
LOCAL STAGING PASSED


## Train YOLO and evaluate quantitative validation pages

Early stopping is expected when validation fitness does not improve for the configured patience period. Evaluation uses `best.pt`.

In [ ]:
model=YOLO(T['model'])
model.train(
    data=str(YAML),
    epochs=T['epochs'],
    patience=T['patience'],
    imgsz=T['image_size'],
    batch=T['batch_size'],
    device=0,
    workers=2,
    project=str(RUNS),
    name=RUN,
    exist_ok=False,
    seed=T['seed'],
    plots=True,
    cache='disk',
)

BEST=RUN_DIR/'weights'/'best.pt'
if not BEST.exists():
    raise RuntimeError('best.pt missing after training.')

trained=YOLO(str(BEST))
metrics=trained.val(
    data=str(YAML),
    split='val',
    imgsz=T['image_size'],
    device=0,
    project=str(RUN_PRED),
    name='metrics',
    exist_ok=True,
)
metric_dict={key:float(value) for key,value in metrics.results_dict.items()}

trained.predict(
    source=str(DATASET/'images'/'val'),
    save=True,
    conf=T['prediction_confidence'],
    imgsz=T['image_size'],
    device=0,
    project=str(RUN_PRED),
    name='validation_tiles',
    exist_ok=True,
    retina_masks=True,
)

for page in sorted(val_pages):
    hits=[p for p in images(ALIGNED) if p.stem.lower()==page.lower()]
    if not hits:
        print('Missing aligned validation page:',page)
        continue
    trained.predict(
        source=str(hits[0]),
        save=True,
        conf=T['prediction_confidence'],
        imgsz=1280,
        device=0,
        project=str(RUN_PRED),
        name='validation_full_pages',
        exist_ok=True,
        retina_masks=True,
    )

print(json.dumps(metric_dict,indent=2))

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/MusicAnnotationProject_local/dataset_v3_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=20261007_171755_yolo11s-s

## GST original/background-normalised/CLAHE comparison

Each GST page is evaluated three ways using the same trained model and overlapping tiles. No binary threshold is applied to the input page because faint pencil could be removed.

In [ ]:
GST_COMPARISON_ROOT=RUN_PRED/'gst_normalisation_comparison'
GST_NORMALISED_DIR=GST_COMPARISON_ROOT/'normalised_pages'
GST_PREDICTION_DIR=GST_COMPARISON_ROOT/'predictions'
GST_COMPARISON_DIR=GST_COMPARISON_ROOT/'comparisons'
for folder in [GST_COMPARISON_ROOT,GST_NORMALISED_DIR,GST_PREDICTION_DIR,GST_COMPARISON_DIR]:
    folder.mkdir(parents=True,exist_ok=True)


def normalise_background(image,blur_sigma=BACKGROUND_BLUR_SIGMA):
    background=cv2.GaussianBlur(image,(0,0),blur_sigma)
    background=np.maximum(background,1)
    return cv2.divide(image,background,scale=255)


def apply_clahe(image):
    clahe=cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,tileGridSize=CLAHE_GRID_SIZE)
    return clahe.apply(image)


def gst_tile_positions(length):
    if length<=GST_TILE_SIZE:
        return [0]
    positions=list(range(0,length-GST_TILE_SIZE+1,GST_TILE_STRIDE))
    final_position=length-GST_TILE_SIZE
    if positions[-1]!=final_position:
        positions.append(final_position)
    return positions


def tiled_gst_predict(gray_image):
    colour_image=cv2.cvtColor(gray_image,cv2.COLOR_GRAY2BGR)
    height,width=gray_image.shape
    padded_height=max(height,GST_TILE_SIZE)
    padded_width=max(width,GST_TILE_SIZE)
    canvas=np.full((padded_height,padded_width,3),255,np.uint8)
    canvas[:height,:width]=colour_image
    confidence_map=np.zeros((padded_height,padded_width),np.float32)
    instance_records=[]

    for y in gst_tile_positions(padded_height):
        for x in gst_tile_positions(padded_width):
            tile=canvas[y:y+GST_TILE_SIZE,x:x+GST_TILE_SIZE]
            result=trained.predict(
                tile,
                conf=GST_CONFIDENCE,
                imgsz=GST_IMAGE_SIZE,
                device=0,
                verbose=False,
                retina_masks=True,
            )[0]
            if result.masks is None:
                continue
            masks=result.masks.data.cpu().numpy()
            confidences=result.boxes.conf.cpu().numpy() if result.boxes is not None else np.ones(len(masks))
            for mask,confidence in zip(masks,confidences):
                mask=cv2.resize(mask,(GST_TILE_SIZE,GST_TILE_SIZE),interpolation=cv2.INTER_NEAREST)>0.5
                mask_pixels=int(np.sum(mask))
                tile_coverage_pct=100*mask_pixels/(GST_TILE_SIZE*GST_TILE_SIZE)
                instance_records.append({
                    'tile_x':int(x),
                    'tile_y':int(y),
                    'confidence':float(confidence),
                    'mask_pixels':mask_pixels,
                    'tile_coverage_pct':tile_coverage_pct,
                })
                region=confidence_map[y:y+GST_TILE_SIZE,x:x+GST_TILE_SIZE]
                update=mask & (confidence>region)
                region[update]=confidence

    confidence_map=confidence_map[:height,:width]
    selected=confidence_map>=GST_CONFIDENCE
    highlighted=colour_image.copy()
    highlighted[selected]=(0,0,255)
    prediction=cv2.addWeighted(colour_image,.68,highlighted,.32,0)
    return prediction,confidence_map,instance_records


gst_results=[]
gst_instance_results=[]

if RUN_GST_NORMALISATION_COMPARISON:
    gst_files=images(GOLD)
    print('GST files:',len(gst_files))
    for source_path in gst_files:
        original=cv2.imread(str(source_path),cv2.IMREAD_GRAYSCALE)
        if original is None:
            print('Unreadable GST file:',source_path)
            continue
        background_normalised=normalise_background(original)
        clahe_normalised=apply_clahe(background_normalised)
        versions={'original':original,'bgnorm':background_normalised,'clahe':clahe_normalised}
        page_normalised_dir=GST_NORMALISED_DIR/source_path.stem
        page_prediction_dir=GST_PREDICTION_DIR/source_path.stem
        page_normalised_dir.mkdir(parents=True,exist_ok=True)
        page_prediction_dir.mkdir(parents=True,exist_ok=True)
        version_outputs={}

        for version_name,page_image in versions.items():
            normalised_path=page_normalised_dir/f'{source_path.stem}_{version_name}.png'
            cv2.imwrite(str(normalised_path),page_image)
            prediction,confidence_map,instance_records=tiled_gst_predict(page_image)
            prediction_path=page_prediction_dir/f'{source_path.stem}_{version_name}_prediction.png'
            cv2.imwrite(str(prediction_path),prediction)
            version_outputs[version_name]=prediction
            selected=confidence_map>=GST_CONFIDENCE
            predicted_pixels=int(np.sum(selected))
            page_pixels=int(page_image.size)
            predicted_page_pct=100*predicted_pixels/page_pixels
            selected_confidences=confidence_map[selected]
            instance_confidences=[record['confidence'] for record in instance_records]
            result_record={
                'file':source_path.name,
                'version':version_name,
                'instances':len(instance_records),
                'predicted_pixels':predicted_pixels,
                'page_pixels':page_pixels,
                'predicted_page_pct':predicted_page_pct,
                'mean_pixel_confidence':float(selected_confidences.mean()) if predicted_pixels else 0.0,
                'maximum_confidence':float(confidence_map.max()),
                'median_instance_confidence':float(np.median(instance_confidences)) if instance_confidences else 0.0,
                'p95_instance_confidence':float(np.percentile(instance_confidences,95)) if instance_confidences else 0.0,
                'largest_instance_tile_pct':max((record['tile_coverage_pct'] for record in instance_records),default=0.0),
                'prediction_file':str(prediction_path),
            }
            gst_results.append(result_record)
            for record in instance_records:
                gst_instance_results.append({'file':source_path.name,'version':version_name,**record})
            print(
                source_path.name,version_name,
                'instances:',len(instance_records),
                'page %:',round(predicted_page_pct,3),
                'max confidence:',round(result_record['maximum_confidence'],4),
                'largest tile mask %:',round(result_record['largest_instance_tile_pct'],3),
            )

        comparison=np.hstack([version_outputs['original'],version_outputs['bgnorm'],version_outputs['clahe']])
        cv2.imwrite(str(GST_COMPARISON_DIR/f'{source_path.stem}_comparison.png'),comparison)
else:
    print('GST normalisation comparison disabled.')

## Display GST normalisation comparisons

Each displayed image is ordered:

1. original-input prediction;
2. background-normalised prediction;
3. background-normalised plus CLAHE prediction.

In [ ]:
if RUN_GST_NORMALISATION_COMPARISON:
    comparison_files=sorted(GST_COMPARISON_DIR.glob('*_comparison.png'))
    for path in comparison_files:
        image=cv2.imread(str(path))
        plt.figure(figsize=(22,12))
        plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        plt.title(path.name+' | original | background normalised | CLAHE')
        plt.axis('off')
        plt.show()

## Export GST diagnostics and combined run summary

In [ ]:
GST_SUMMARY_JSON=GST_COMPARISON_ROOT/'normalisation_summary.json'
GST_SUMMARY_CSV=GST_COMPARISON_ROOT/'normalisation_summary.csv'
GST_INSTANCE_CSV=GST_COMPARISON_ROOT/'gst_instance_predictions.csv'

if RUN_GST_NORMALISATION_COMPARISON:
    GST_SUMMARY_JSON.write_text(json.dumps({
        'model':str(BEST),
        'confidence':GST_CONFIDENCE,
        'tile_size':GST_TILE_SIZE,
        'tile_stride':GST_TILE_STRIDE,
        'background_blur_sigma':BACKGROUND_BLUR_SIGMA,
        'clahe_clip_limit':CLAHE_CLIP_LIMIT,
        'results':gst_results,
    },indent=2))

    with open(GST_SUMMARY_CSV,'w',newline='',encoding='utf-8') as csv_file:
        if gst_results:
            writer=csv.DictWriter(csv_file,fieldnames=list(gst_results[0].keys()))
            writer.writeheader(); writer.writerows(gst_results)

    with open(GST_INSTANCE_CSV,'w',newline='',encoding='utf-8') as csv_file:
        if gst_instance_results:
            writer=csv.DictWriter(csv_file,fieldnames=list(gst_instance_results[0].keys()))
            writer.writeheader(); writer.writerows(gst_instance_results)

summary={
    'run_name':RUN,
    'model':T['model'],
    'training_pages':sorted(train_pages),
    'validation_pages':sorted(val_pages),
    'training_page_count':len(train_pages),
    'validation_page_count':len(val_pages),
    'training_tile_count':len(images(DATASET/'images'/'train')),
    'validation_tile_count':len(images(DATASET/'images'/'val')),
    'metrics':metric_dict,
    'manifest_md5':md5(MANIFEST),
    'best_model':str(BEST),
    'gst_normalisation_enabled':RUN_GST_NORMALISATION_COMPARISON,
    'gst_results':gst_results,
    'gst_instance_csv':str(GST_INSTANCE_CSV) if RUN_GST_NORMALISATION_COMPARISON else None,
}

(RUN_DIR/'run_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Run summary:',RUN_DIR/'run_summary.json')
if RUN_GST_NORMALISATION_COMPARISON:
    print('GST comparisons:',GST_COMPARISON_DIR)
    print('GST page summary:',GST_SUMMARY_CSV)
    print('GST instance diagnostics:',GST_INSTANCE_CSV)

## Copy completed outputs back to Google Drive

Colab local storage is temporary. This final step copies the complete local training run and prediction run to their durable Google Drive folders.

In [ ]:
if STAGE_DATASET_LOCALLY:
    drive_run_dir=DRIVE_RUNS/RUN
    drive_prediction_dir=DRIVE_PREDS/RUN
    if drive_run_dir.exists(): shutil.rmtree(drive_run_dir)
    if drive_prediction_dir.exists(): shutil.rmtree(drive_prediction_dir)
    shutil.copytree(RUN_DIR,drive_run_dir)
    shutil.copytree(RUN_PRED,drive_prediction_dir)
    print('Training run copied to:',drive_run_dir)
    print('Predictions copied to:',drive_prediction_dir)
    print('Best model copied to:',drive_run_dir/'weights'/'best.pt')